# Exame B – Etapa 1: Exploração dos Dados de Bicicletas Compartilhadas de Seul

**Dupla:** Theo Machado Silveira Mello e Claudio Henrique Lopes Mendes

## Resumo

**Objetivo.** Conhecer o dataset `bike_sharing.csv` (registros horários de dez/2017 a nov/2018) e
entender **o que faz a demanda (`Rented_Bike_Count`) subir ou cair**, deixando decisões prontas
para os testes de hipótese (Etapa 2) e para a regressão linear (Etapa 3).

**O que foi feito.** Verificamos a qualidade dos dados, propusemos e aplicamos os tratamentos,
descrevemos cada variável isoladamente e depois analisamos como cada uma se relaciona com a demanda.

**Principais achados.**

1. **Os dados têm erros que não aparecem como valores ausentes.** Há 20 registros com um zero
   impossível em `Humidity`, `Dew_Point_Temperature` ou `Temperature`. Uma relação física entre as
   três variáveis (fórmula de Magnus) permitiu localizar e corrigir esses valores.
2. **Sistema fechado significa demanda zero.** As 234 horas com `Functioning_Day = No` têm
   exatamente 0 aluguéis, e nenhuma hora de operação tem 0. Portanto, essas horas não dizem nada
   sobre a preferência dos usuários e foram separadas da análise de demanda.
3. **Hora e tipo de dia precisam ser analisados juntos.** Dias úteis têm dois picos de deslocamento
   (8h e 18h), que desaparecem nos fins de semana e feriados. Comparado só pelas
   distribuições gerais, o tipo de dia parece pouco importante, mas ele muda o formato do dia.
4. **Temperatura é a variável contínua mais forte, mas a relação não é linear:** a demanda cresce
   até cerca de 30 °C e depois cai.
5. **Chuva derruba a demanda:** a mediana cai de 591 para 70 bicicletas por hora, algo que a
   correlação de Pearson (−0,14) esconde.
6. **O efeito bruto do feriado engana.** A queda aparente vem, em boa parte, de os feriados se
   concentrarem no inverno; no verão e no outono, as medianas são praticamente iguais.

As recomendações para as etapas seguintes estão na **Seção 7**.

## Seção 1 – Configuração do Ambiente

Importamos as bibliotecas, mostramos as versões usadas e fixamos a semente aleatória, conforme as
orientações do exame.

In [ ]:
# Objetivo: Importar as bibliotecas, mostrar as versões e fixar a semente aleatória.
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import statsmodels
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

RANDOM_STATE = 2026
np.random.seed(RANDOM_STATE)

for biblioteca in (pd, np, scipy, matplotlib, sns, statsmodels):
    print(f"{biblioteca.__name__:<12}{biblioteca.__version__}")

: 

In [ ]:
# Objetivo: Definir as constantes do projeto e o estilo comum de todos os gráficos.
ALVO = "Rented_Bike_Count"
METEOROLOGICAS = [
    "Temperature", "Humidity", "Wind_Speed", "Visibility",
    "Dew_Point_Temperature", "Solar_Radiation", "Rainfall", "Snowfall",
]
NUMERICAS = [ALVO, "Hour"] + METEOROLOGICAS
ORDEM_ESTACOES = ["Winter", "Spring", "Summer", "Autumn"]
ORDEM_TIPO_DIA = ["Dia útil", "Fim de semana", "Feriado"]
ORDEM_MESES = [12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

AZUL, LARANJA, VERDE = "#2a78d6", "#eb6834", "#1baf7a"
CINZA = "#898781"
TINTA = "#0b0b0b"
TINTA_SECUNDARIA = "#52514e"
SUPERFICIE = "#fcfcfb"
CORES_TIPO_DIA = dict(zip(ORDEM_TIPO_DIA, [AZUL, LARANJA, VERDE]))

MAPA_SEQUENCIAL = LinearSegmentedColormap.from_list(
    "azul", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
MAPA_DIVERGENTE = LinearSegmentedColormap.from_list(
    "vermelho_azul", ["#e34948", "#f0efec", AZUL]
)

plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": SUPERFICIE,
    "axes.facecolor": SUPERFICIE,
    "axes.edgecolor": "#c3c2b7",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.7,
    "axes.titlesize": 11,
    "axes.titlelocation": "left",
    "axes.labelsize": 9.5,
    "axes.labelcolor": TINTA_SECUNDARIA,
    "text.color": TINTA,
    "xtick.color": TINTA_SECUNDARIA,
    "ytick.color": TINTA_SECUNDARIA,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "lines.linewidth": 2,
    "legend.frameon": False,
    "legend.fontsize": 9,
})

## Seção 2 – Carregamento e Visão Geral

### 2.1 Leitura do arquivo

A função abaixo procura o arquivo localmente e, se não o encontrar, baixa do repositório da
disciplina. Assim, o notebook roda no Google Colab sem precisar de upload manual.

In [ ]:
# Objetivo: Definir a função de leitura e carregar o dataset de treino e o exemplo do teste.
PASTA_EXAME = "exames/exame-b-bike-sharing"
URL_BASE = f"https://raw.githubusercontent.com/glaucogoncalves/p-e/main/{PASTA_EXAME}"


def carregar_dados(nome_arquivo):
    """Lê um CSV do exame, procurando primeiro em pastas locais e depois no GitHub.

    Parâmetros:
        nome_arquivo (str): nome do arquivo, por exemplo "bike_sharing.csv".

    Retorna:
        pandas.DataFrame com o conteúdo do arquivo.
    """
    for pasta in (Path("."), Path("../p-e") / PASTA_EXAME):
        if (pasta / nome_arquivo).exists():
            return pd.read_csv(pasta / nome_arquivo)
    return pd.read_csv(f"{URL_BASE}/{nome_arquivo}")


df_bruto = carregar_dados("bike_sharing.csv")
df_exemplo_teste = carregar_dados("exemplo.csv")

print("Shape:", df_bruto.shape)
display(df_bruto.head())

### 2.2 Estrutura das variáveis

São 7.008 linhas e 14 colunas, como informa o `data_description.md`. Cada linha representa uma hora
de um dia. Conferimos abaixo se os tipos lidos pelo pandas correspondem ao significado de cada
coluna.

In [ ]:
# Objetivo: Listar o tipo de cada coluna, a quantidade de valores distintos e um exemplo.
estrutura = pd.DataFrame({
    "tipo": df_bruto.dtypes.astype(str),
    "valores_distintos": df_bruto.nunique(),
    "exemplo": df_bruto.iloc[0],
})
display(estrutura)

**O que observamos:** `Date` foi lida como texto e precisa ser convertida para data (Seção 4).
`Hour` é numérica, mas funciona como categoria: a hora 23 não é "maior" que a hora 8 em nenhum
sentido útil para a demanda. Isso será confirmado na Seção 6.

### 2.3 Cobertura temporal e relação com o teste cego

O período tem 365 dias, ou seja, 8.760 horas possíveis. Vamos verificar quais horas faltam e como
isso se relaciona com o arquivo de teste.

In [ ]:
# Objetivo: Medir a cobertura temporal e verificar se o exemplo do teste aparece no treino.
datas = pd.to_datetime(df_bruto["Date"], format="%d/%m/%Y")
horas_possiveis = datas.nunique() * 24
horas_por_dia = df_bruto.groupby("Date").size()
horas_por_faixa = df_bruto["Hour"].value_counts()
chaves_treino = set(zip(df_bruto["Date"], df_bruto["Hour"]))
chaves_exemplo = set(zip(df_exemplo_teste["Date"], df_exemplo_teste["Hour"]))

print(f"Período: {datas.min():%d/%m/%Y} a {datas.max():%d/%m/%Y} ({datas.nunique()} dias)")
print(f"Horas presentes: {len(df_bruto)} de {horas_possiveis} "
      f"({len(df_bruto) / horas_possiveis:.1%})")
print(f"Horas registradas por dia: mín. {horas_por_dia.min()}, máx. {horas_por_dia.max()}")
print(f"Registros por hora do dia: mín. {horas_por_faixa.min()}, máx. {horas_por_faixa.max()}")
print(f"Linhas de exemplo.csv presentes no treino: {len(chaves_exemplo & chaves_treino)} de "
      f"{len(chaves_exemplo)}")

**O que observamos:** temos exatamente 80% das horas do ano, com faltas espalhadas por todos os
dias e por todas as horas do dia. As linhas de `exemplo.csv` estão entre as horas que faltam.

**Conclusão provável:** o teste cego são os 20% restantes, sorteados entre as horas do mesmo ano.
Não podemos confirmar isso, pois só vemos 5 linhas do teste, mas há duas consequências práticas:

- As horas ausentes não são um problema de qualidade, e sim parte do desenho do exame.
- Na Etapa 3, uma validação com separação aleatória imita bem esse cenário, porque o teste também
  mistura horas dos mesmos dias do treino.

## Seção 3 – Qualidade dos Dados

### 3.1 Valores ausentes e duplicados

In [ ]:
# Objetivo: Contar valores ausentes, linhas duplicadas e chaves (Date, Hour) repetidas.
print("Valores ausentes no dataset inteiro:", df_bruto.isna().sum().sum())
print("Linhas totalmente duplicadas:", df_bruto.duplicated().sum())
print("Pares (Date, Hour) repetidos:", df_bruto.duplicated(subset=["Date", "Hour"]).sum())

**O que observamos:** não há ausentes nem duplicados. Isso não garante que os dados estejam
corretos: valores errados podem estar preenchidos normalmente, como veremos em 3.3.

### 3.2 Consistência do calendário

Verificamos se as colunas de calendário concordam com a data e se a radiação solar é nula à noite.

In [ ]:
# Objetivo: Testar a coerência entre data, estação, feriado, funcionamento e radiação noturna.
por_dia = df_bruto.groupby("Date")
periodo_estacoes = (
    df_bruto.assign(data=datas).groupby("Seasons")["data"].agg(["min", "max"])
    .loc[ORDEM_ESTACOES]
)
display(periodo_estacoes)

noite = df_bruto["Hour"].isin([21, 22, 23, 0, 1, 2, 3, 4, 5])
print("Dias com mais de uma estação:", (por_dia["Seasons"].nunique() > 1).sum())
print("Dias com feriado em apenas parte das horas:", (por_dia["Holiday"].nunique() > 1).sum())
print("Horas noturnas (21h a 5h) com radiação solar > 0:",
      (df_bruto.loc[noite, "Solar_Radiation"] > 0).sum())

dias_mistos = por_dia["Functioning_Day"].nunique().loc[lambda s: s > 1].index
print("\nDias com funcionamento parcial:", list(dias_mistos))
display(
    df_bruto[df_bruto["Date"].isin(dias_mistos)]
    .groupby(["Date", "Functioning_Day"])["Hour"].agg(["min", "max", "count"])
)

**O que observamos:** o calendário é coerente. As estações seguem os meses do ano no hemisfério
norte, e não há radiação solar à noite. O único caso especial é **06/10/2018**, quando o sistema
ficou fechado até 6h e funcionou a partir das 8h. Por isso, `Functioning_Day` deve ser tratada hora
a hora, e não por dia.

### 3.3 Valores fisicamente impossíveis: o teste de Magnus

O mínimo de 0% em `Humidity` chama atenção: umidade relativa nula não ocorre na atmosfera real.
Como temperatura, ponto de orvalho e umidade relativa estão ligados por uma relação física, podemos
usar a **fórmula de Magnus** para testar a coerência de cada linha:

$$UR \approx 100 \cdot \exp\left(\frac{a\,T_d}{b + T_d} - \frac{a\,T}{b + T}\right),
\quad a = 17{,}62,\; b = 243{,}12$$

em que $T$ é a temperatura e $T_d$ é o ponto de orvalho, em °C. Se a fórmula concordar com a umidade
informada em quase todas as linhas, as linhas que discordam indicam erro de registro.

In [ ]:
# Objetivo: Definir a fórmula de Magnus e comparar a umidade calculada com a informada.
MAGNUS_A, MAGNUS_B = 17.62, 243.12
LIMITE_ERRO_UMIDADE = 5


def umidade_magnus(temperatura, ponto_orvalho):
    """Calcula a umidade relativa (%) pela fórmula de Magnus.

    Parâmetros:
        temperatura (Series): temperatura do ar, em °C.
        ponto_orvalho (Series): temperatura do ponto de orvalho, em °C.

    Retorna:
        Series com a umidade relativa estimada.
    """
    termo_orvalho = MAGNUS_A * ponto_orvalho / (MAGNUS_B + ponto_orvalho)
    termo_ar = MAGNUS_A * temperatura / (MAGNUS_B + temperatura)
    return 100 * np.exp(termo_orvalho - termo_ar)


umidade_calculada = umidade_magnus(df_bruto["Temperature"], df_bruto["Dew_Point_Temperature"])
erro_umidade = (umidade_calculada - df_bruto["Humidity"]).abs()
inconsistente = erro_umidade > LIMITE_ERRO_UMIDADE

maiores_erros = erro_umidade.sort_values(ascending=False).round(2).to_list()
print("Linhas com erro acima de", LIMITE_ERRO_UMIDADE, "pontos:", inconsistente.sum())
print("Menor erro entre as inconsistentes:", maiores_erros[inconsistente.sum() - 1])
print("Maior erro entre as demais:", maiores_erros[inconsistente.sum()])

In [ ]:
# Objetivo: Comparar graficamente a umidade informada com a umidade calculada por Magnus.
fig, ax = plt.subplots(figsize=(6.2, 4.6))
ax.scatter(df_bruto.loc[~inconsistente, "Humidity"], umidade_calculada[~inconsistente],
           s=5, color=CINZA, alpha=0.35, label=f"Coerentes ({(~inconsistente).sum()} linhas)")
ax.scatter(df_bruto.loc[inconsistente, "Humidity"], umidade_calculada[inconsistente],
           s=32, color=LARANJA, edgecolor=SUPERFICIE, linewidth=0.8,
           label=f"Inconsistentes ({inconsistente.sum()} linhas)", zorder=3)
ax.set(xlabel="Umidade informada (%)", ylabel="Umidade calculada por Magnus (%)",
       title="Só 20 linhas fogem da relação física entre as três variáveis")
ax.legend(loc="upper left")
plt.show()

In [ ]:
# Objetivo: Mostrar as linhas inconsistentes e identificar qual variável está com o zero.
colunas_fisicas = ["Temperature", "Humidity", "Dew_Point_Temperature"]
casos = df_bruto.loc[inconsistente, ["Date", "Hour"] + colunas_fisicas].copy()
casos["Umidade_Magnus"] = umidade_calculada[inconsistente].round(1)
casos["Coluna_com_zero"] = (casos[colunas_fisicas] == 0).idxmax(axis=1)
display(casos.sort_values(["Coluna_com_zero", "Date", "Hour"]))
print(casos["Coluna_com_zero"].value_counts().to_string())

**O que observamos:**

- A separação é muito nítida: nas 6.988 linhas coerentes, o erro nunca passa de 0,7 ponto
  percentual; nas 20 inconsistentes, ele é sempre maior que 9 pontos. Por isso, o limite de 5
  pontos não é uma escolha sensível.
- **Todas** as 20 linhas têm um zero em uma das três variáveis: 10 em `Humidity`, 9 em
  `Dew_Point_Temperature` e 1 em `Temperature`.
- As ocorrências se agrupam no tempo: quase todas acontecem entre 19/05 e 02/06/2018, o que sugere
  uma falha temporária do sensor ou do registro, com o zero ocupando o lugar do valor ausente.
- Esses zeros passam despercebidos em `describe()` e em `isna()`. Nem todo zero é erro: outras
  linhas têm 0,0 °C em `Temperature` ou `Dew_Point_Temperature` e são perfeitamente coerentes. Sem
  o teste, removê-las seria um erro.

### 3.4 Outros valores suspeitos e formatos especiais

In [ ]:
# Objetivo: Investigar neve em temperatura alta, chuva abaixo de 0 °C e os casos mais extremos.
neve = df_bruto[df_bruto["Snowfall"] > 0]
chuva = df_bruto[df_bruto["Rainfall"] > 0]
neve_quente = neve[neve["Temperature"] > 5]

print(f"Horas com neve: {len(neve)}; com temperatura acima de 5 °C: {len(neve_quente)}")
print("Maior temperatura com neve registrada:", neve["Temperature"].max(), "°C")
print(f"Horas com chuva: {len(chuva)}; com temperatura abaixo de 0 °C: "
      f"{(chuva['Temperature'] < 0).sum()}")
print("Estações com neve:", sorted(neve["Seasons"].unique()))

colunas_neve = ["Date", "Hour", "Temperature", "Snowfall", "Solar_Radiation", ALVO]
print("\nMaior volume de neve da série:")
display(neve.nlargest(1, "Snowfall")[colunas_neve])
print("Horas com neve e as temperaturas mais altas:")
display(neve.nlargest(6, "Temperature")[colunas_neve])

In [ ]:
# Objetivo: Medir a concentração de valores em um único ponto em cada variável meteorológica.
def medir_concentracao(df, colunas):
    """Calcula o valor mais frequente de cada coluna e sua participação nas linhas.

    Parâmetros:
        df (DataFrame): dados de origem.
        colunas (list): nomes das colunas numéricas a analisar.

    Retorna:
        DataFrame com o valor mais frequente e o percentual de linhas nesse valor.
    """
    linhas = []
    for coluna in colunas:
        moda = df[coluna].mode().iloc[0]
        linhas.append({
            "variavel": coluna,
            "valor_mais_frequente": moda,
            "pct_linhas": round(100 * (df[coluna] == moda).mean(), 1),
        })
    return pd.DataFrame(linhas).set_index("variavel").sort_values("pct_linhas", ascending=False)


display(medir_concentracao(df_bruto, METEOROLOGICAS))

**O que observamos:**

- **`Snowfall` pode medir a neve acumulada, e não a que está caindo.** Há registros de neve a mais
  de 5 °C e sob radiação solar, como em 25/11/2018, um dia depois do maior valor da série. A
  descrição oficial não define isso, então é uma interpretação nossa. Os valores foram mantidos.
- **`Rainfall` e `Snowfall` são quase sempre zero** (cerca de 94% e 95% das linhas). Mais do que a
  quantidade, o sinal útil pode ser a simples ocorrência, o que motiva criar indicadores binários.
- **`Visibility` tem um teto em 2.000** (cerca de 25% das linhas), provavelmente o limite do
  instrumento. Acima dele, a visibilidade real é desconhecida.
- **`Solar_Radiation` é zero em quase metade das linhas**, simplesmente porque metade das horas é
  noite. O zero, neste caso, é legítimo.
- A chuva em temperatura negativa aparece em poucas horas (13) e não muda as conclusões. Ela fica
  registrada como ponto de atenção, sem tratamento.

### 3.5 Quadro-resumo e propostas de tratamento

| Problema | Linhas | Proposta |
|---|---:|---|
| Zeros impossíveis em `Humidity`, `Dew_Point_Temperature`, `Temperature` | 20 | **Corrigir** usando a fórmula de Magnus |
| Sistema fechado (`Functioning_Day = No`) com demanda zero | 234 | **Separar** da análise de demanda, sem apagar do dataset |
| `Date` como texto | todas | **Converter** e derivar variáveis de calendário |
| `Rainfall` e `Snowfall` quase sempre zero | ~94–95% | **Criar** indicadores binários |
| `Visibility` limitada em 2.000 | ~25% | **Manter** e registrar a limitação |
| `Snowfall` possivelmente acumulada | – | **Manter** e registrar a interpretação |

## Seção 4 – Tratamento dos Dados

Todo o tratamento fica em funções que **não dependem de `Rented_Bike_Count`**, para que possam ser
reaplicadas sem alteração ao arquivo de teste cego na Etapa 3.

**Justificativa – correção dos zeros impossíveis.** Temperatura, ponto de orvalho e umidade estão
ligados por uma relação física, e a seção 3.3 mostrou que a fórmula acerta com erro inferior a 0,7
ponto nas 6.988 linhas válidas. Logo, é possível reconstruir o valor errado a partir dos outros
dois, em vez de perder a linha inteira ou preencher com uma média, que ignoraria a hora e o clima
daquele momento. Só são alteradas linhas que **falham no teste e têm um zero**. O teste cego pode
conter o mesmo tipo de erro, e essa regra funciona lá também.

In [ ]:
# Objetivo: Definir as funções inversas de Magnus e a correção dos zeros impossíveis.
def orvalho_magnus(temperatura, umidade):
    """Calcula o ponto de orvalho (°C) a partir da temperatura (°C) e da umidade (%).

    Parâmetros:
        temperatura (Series): temperatura do ar, em °C.
        umidade (Series): umidade relativa, em %, maior que zero.

    Retorna:
        Series com o ponto de orvalho estimado.
    """
    gama = np.log(umidade / 100) + MAGNUS_A * temperatura / (MAGNUS_B + temperatura)
    return MAGNUS_B * gama / (MAGNUS_A - gama)


def temperatura_magnus(ponto_orvalho, umidade):
    """Calcula a temperatura (°C) a partir do ponto de orvalho (°C) e da umidade (%).

    Parâmetros:
        ponto_orvalho (Series): temperatura do ponto de orvalho, em °C.
        umidade (Series): umidade relativa, em %, maior que zero.

    Retorna:
        Series com a temperatura estimada.
    """
    termo = MAGNUS_A * ponto_orvalho / (MAGNUS_B + ponto_orvalho) - np.log(umidade / 100)
    return MAGNUS_B * termo / (MAGNUS_A - termo)


def corrigir_zeros_impossiveis(df):
    """Reconstrói, por Magnus, os zeros que violam a relação física entre as três variáveis.

    Parâmetros:
        df (DataFrame): dados com Temperature, Humidity e Dew_Point_Temperature.

    Retorna:
        Novo DataFrame com os valores corrigidos; os demais valores ficam inalterados.
    """
    saida = df.copy()
    temp, umid, orvalho = (
        df["Temperature"], df["Humidity"], df["Dew_Point_Temperature"]
    )
    falha = (umidade_magnus(temp, orvalho) - umid).abs() > LIMITE_ERRO_UMIDADE
    umid_zero = falha & (umid == 0)
    orvalho_zero = falha & (orvalho == 0) & ~umid_zero
    temp_zero = falha & (temp == 0) & ~umid_zero & ~orvalho_zero

    saida.loc[umid_zero, "Humidity"] = (
        umidade_magnus(temp[umid_zero], orvalho[umid_zero]).round().astype(int)
    )
    saida.loc[orvalho_zero, "Dew_Point_Temperature"] = (
        orvalho_magnus(temp[orvalho_zero], umid[orvalho_zero]).round(1)
    )
    saida.loc[temp_zero, "Temperature"] = (
        temperatura_magnus(orvalho[temp_zero], umid[temp_zero]).round(1)
    )
    return saida

**Justificativa – variáveis de calendário e indicadores.** O `data_description.md` orienta a não
usar `Date` diretamente, mas permite derivar variáveis dela. O mês e o dia da semana capturam
hábitos que as colunas originais não mostram, como o deslocamento para o trabalho. `Day_Type`
organiza os dias em três grupos: feriado tem prioridade sobre fim de semana, pois já são dias sem a
rotina de trabalho. `Is_Rainy` e `Has_Snow` registram a simples ocorrência, porque as duas variáveis
originais são quase sempre zero (seção 3.4).

In [ ]:
# Objetivo: Definir a criação de variáveis de calendário e o fluxo completo de preparação.
def criar_variaveis_derivadas(df):
    """Cria variáveis de calendário e indicadores de chuva e neve.

    Parâmetros:
        df (DataFrame): dados com Date, Holiday, Rainfall e Snowfall.

    Retorna:
        Novo DataFrame com Date convertida e as colunas Month, Day_Of_Week, Is_Weekend,
        Day_Type, Is_Rainy e Has_Snow.
    """
    saida = df.copy()
    saida["Date"] = pd.to_datetime(saida["Date"], format="%d/%m/%Y")
    saida["Month"] = saida["Date"].dt.month
    saida["Day_Of_Week"] = saida["Date"].dt.day_of_week
    saida["Is_Weekend"] = (saida["Day_Of_Week"] >= 5).astype(int)
    saida["Day_Type"] = np.select(
        [saida["Holiday"] == "Holiday", saida["Is_Weekend"] == 1],
        ["Feriado", "Fim de semana"],
        default="Dia útil",
    )
    saida["Is_Rainy"] = (saida["Rainfall"] > 0).astype(int)
    saida["Has_Snow"] = (saida["Snowfall"] > 0).astype(int)
    return saida


def preparar_dados(df):
    """Aplica o fluxo de tratamento completo, sem usar a variável resposta.

    Parâmetros:
        df (DataFrame): dados no formato original (treino ou teste cego).

    Retorna:
        Novo DataFrame tratado, com o mesmo número de linhas da entrada.
    """
    return criar_variaveis_derivadas(corrigir_zeros_impossiveis(df))


df = preparar_dados(df_bruto)

In [ ]:
# Objetivo: Mostrar o dataset antes e depois da correção e conferir o resultado da fórmula.
print("Shape antes:", df_bruto.shape, "| Shape depois:", df.shape)
print("Colunas novas:", [c for c in df.columns if c not in df_bruto.columns])
print("Missing após o tratamento:", df.isna().sum().sum())

colunas_fisicas = ["Temperature", "Humidity", "Dew_Point_Temperature"]
print("\nEstatísticas ANTES da correção:")
display(df_bruto[colunas_fisicas].describe().T[["mean", "min", "max"]].round(2))
print("Estatísticas DEPOIS da correção:")
display(df[colunas_fisicas].describe().T[["mean", "min", "max"]].round(2))

erro_final = (umidade_magnus(df["Temperature"], df["Dew_Point_Temperature"]) - df["Humidity"]).abs()
print("Linhas ainda inconsistentes:", (erro_final > LIMITE_ERRO_UMIDADE).sum())
print("Células alteradas:", (df[colunas_fisicas] != df_bruto[colunas_fisicas]).sum().sum())

In [ ]:
# Objetivo: Validar a correção da temperatura comparando com as horas vizinhas do mesmo dia.
vizinhas = (df["Date"] == "2018-11-29") & df["Hour"].between(16, 20)
validacao = df.loc[vizinhas, ["Hour", "Humidity", "Dew_Point_Temperature"]].copy()
validacao["Temperature_original"] = df_bruto.loc[vizinhas, "Temperature"]
validacao["Temperature_corrigida"] = df.loc[vizinhas, "Temperature"]
display(validacao.sort_values("Hour"))

**O que observamos:** a correção alterou exatamente 20 células, e nenhuma linha continua
inconsistente. Há uma validação independente: a temperatura zerada às 18h de 29/11/2018 foi
reconstruída como **6,0 °C**, valor coerente com as horas vizinhas (5,8 °C às 17h e 5,2 °C às 19h),
que **não** foram usadas no cálculo.

Também conferimos que o fluxo funciona em um arquivo no formato do teste, sem a coluna-alvo:

In [ ]:
# Objetivo: Conferir que a preparação funciona no formato do teste cego, sem a coluna-alvo.
exemplo_preparado = preparar_dados(df_exemplo_teste)
print("Shape antes:", df_exemplo_teste.shape, "| Shape depois:", exemplo_preparado.shape)
print("Contém a coluna-alvo?", ALVO in exemplo_preparado.columns)
display(exemplo_preparado[["Date", "Hour", "Humidity", "Month", "Day_Type", "Is_Rainy"]])

**Justificativa – separar as horas com o sistema fechado.** A primeira tabela abaixo mostra que
`Functioning_Day = No` determina demanda zero, sem nenhuma exceção. Essas horas não informam quanto as pessoas querem pedalar, e sim que não
havia bicicletas disponíveis. Mantê-las misturaria dois fenômenos e distorceria as relações com
clima e calendário, sobretudo no outono, onde está a maioria desses casos. Por isso criamos
`df_op`, usado na análise de demanda. O `df` completo **continua guardado**: o teste cego pode
conter horas com o sistema fechado, e para elas a previsão correta é zero.

In [ ]:
# Objetivo: Criar o recorte de operação e comparar o alvo com e sem as horas fechadas.
df_op = df[df["Functioning_Day"] == "Yes"].copy()

print("Shape do df completo:", df.shape, "| Shape do df_op:", df_op.shape)
display(pd.crosstab(df["Functioning_Day"], df[ALVO] == 0,
                    rownames=["Functioning_Day"], colnames=["Demanda = 0"]))
display(pd.crosstab(df["Seasons"], df["Functioning_Day"]).loc[ORDEM_ESTACOES])

comparacao = pd.DataFrame({
    "Todas as horas": df[ALVO].describe(),
    "Horas de operação": df_op[ALVO].describe(),
}).round(1)
display(comparacao)

**Justificativa – manter os valores extremos.** Não removemos outliers nesta etapa. A tabela abaixo
mostra que as cinco maiores demandas são às 18h de dias úteis de junho, e que três dos cinco maiores
volumes de chuva são de agosto, com demanda baixa: são situações plausíveis e informativas, e não
erros de medição. Removê-las ocultaria justamente o comportamento que queremos entender. Na Etapa
3, a influência desses pontos poderá ser reavaliada com a análise de resíduos.

In [ ]:
# Objetivo: Listar os registros com os maiores valores de demanda, chuva e vento.
colunas_contexto = ["Date", "Hour", "Day_Type", "Temperature", ALVO]
for coluna in [ALVO, "Rainfall", "Wind_Speed"]:
    colunas_tabela = list(dict.fromkeys(colunas_contexto + [coluna]))
    print(f"\nMaiores valores de {coluna}:")
    display(df.nlargest(5, coluna)[colunas_tabela])

## Seção 5 – Análise Univariada

A partir daqui, a análise usa as horas de operação (`df_op`), salvo quando indicado.

### 5.1 Medidas resumo das variáveis numéricas

Além de média, mediana, desvio padrão e assimetria, a tabela conta os *outliers* pelo critério do
*boxplot* visto em aula (mais de 1,5 × IQR além dos quartis).

In [ ]:
# Objetivo: Calcular as medidas resumo e contar outliers pelo critério do intervalo interquartil.
def resumir_numericas(df, colunas):
    """Calcula medidas de localização, escala, forma e contagem de outliers.

    Parâmetros:
        df (DataFrame): dados de origem.
        colunas (list): nomes das colunas numéricas.

    Retorna:
        DataFrame com uma linha por variável.
    """
    linhas = {}
    for coluna in colunas:
        serie = df[coluna]
        q1, q3 = serie.quantile([0.25, 0.75])
        iqr = q3 - q1
        fora = (serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)
        linhas[coluna] = {
            "média": serie.mean(), "mediana": serie.median(), "desvio_padrão": serie.std(),
            "mín": serie.min(), "Q1": q1, "Q3": q3, "máx": serie.max(), "IQR": iqr,
            "assimetria": serie.skew(), "curtose": serie.kurt(), "outliers_iqr": fora.sum(),
        }
    return pd.DataFrame(linhas).T


display(resumir_numericas(df_op, NUMERICAS).round(2))

### 5.2 Histogramas e *boxplots*

In [ ]:
# Objetivo: Definir uma função que desenha histograma e boxplot de várias variáveis em grade.
def plotar_distribuicoes(df, colunas, escala_log=(), n_colunas=3):
    """Desenha, para cada variável, um histograma com um boxplot logo abaixo.

    Parâmetros:
        df (DataFrame): dados de origem.
        colunas (list): variáveis numéricas a plotar.
        escala_log (iterable): variáveis cujo histograma usa eixo vertical logarítmico.
        n_colunas (int): quantidade de colunas da grade.

    Retorna:
        Figura do matplotlib.
    """
    n_linhas = int(np.ceil(len(colunas) / n_colunas))
    fig = plt.figure(figsize=(4.1 * n_colunas, 3.0 * n_linhas), constrained_layout=True)
    grades = fig.add_gridspec(n_linhas, n_colunas)
    for grade, coluna in zip(grades, colunas):
        sub = grade.subgridspec(2, 1, height_ratios=[4, 1], hspace=0.04)
        eixo_hist, eixo_box = fig.add_subplot(sub[0]), fig.add_subplot(sub[1])
        eixo_hist.hist(df[coluna], bins=30, color=AZUL, edgecolor=SUPERFICIE, linewidth=0.5)
        if coluna in escala_log:
            eixo_hist.set_yscale("log")
        sufixo = " (freq. em log)" if coluna in escala_log else ""
        eixo_hist.set(title=f"{coluna}{sufixo}", ylabel="Horas")
        eixo_hist.tick_params(labelbottom=False)
        sns.boxplot(x=df[coluna], ax=eixo_box, color="#b7d3f6", linewidth=0.8, fliersize=1.5)
        eixo_box.sharex(eixo_hist)
        eixo_box.set(xlabel="", yticks=[])
        eixo_box.grid(False)
    return fig


plotar_distribuicoes(df_op, [ALVO] + METEOROLOGICAS, escala_log=("Rainfall", "Snowfall"))
plt.show()

**O que observamos em cada variável:**

| Variável | Forma | Leitura |
|---|---|---|
| `Rented_Bike_Count` | Assimétrica à direita (1,14) | Média (729) acima da mediana (540); cauda longa até 3.556 |
| `Temperature` | Quase simétrica, um pouco achatada | Larga amplitude: de −17,8 °C a 39,4 °C |
| `Dew_Point_Temperature` | Leve assimetria à esquerda | Forma muito parecida com a da temperatura, o que antecipa a redundância entre as duas |
| `Humidity` | Aproximadamente simétrica | Depois da correção, o mínimo passou a ser plausível |
| `Wind_Speed` | Assimétrica à direita | Maioria das horas com vento fraco; quatro dos cinco maiores valores são de 10/04/2018 |
| `Visibility` | Assimétrica à esquerda | Acúmulo no teto de 2.000 |
| `Solar_Radiation` | Muito assimétrica à direita | Grande massa no zero, correspondente às horas noturnas |
| `Rainfall`, `Snowfall` | Extremamente assimétricas | Quase tudo em zero; os "outliers" pelo IQR são simplesmente todas as horas com chuva ou neve |

**Cuidado com a regra do IQR:** quando mais de 75% dos valores são zero, o IQR também é zero, e
*qualquer* chuva vira "outlier". Por isso, a contagem da tabela não pode ser usada automaticamente
como critério de remoção.

### 5.3 A variável resposta e possíveis transformações

A regressão linear será ajustada na Etapa 3, e a forma de `Rented_Bike_Count` influencia essa
escolha. Comparamos a escala original com raiz quadrada e logaritmo, usando o *QQ-plot* visto em
aula para avaliar a proximidade da distribuição normal.

In [ ]:
# Objetivo: Comparar histogramas e QQ-plots do alvo em três escalas diferentes.
transformacoes = {
    "Original": df_op[ALVO],
    "Raiz quadrada": np.sqrt(df_op[ALVO]),
    "Logaritmo (log1p)": np.log1p(df_op[ALVO]),
}

fig, eixos = plt.subplots(2, 3, figsize=(11.5, 5.6), constrained_layout=True)
for (nome, serie), eixo_hist, eixo_qq in zip(transformacoes.items(), eixos[0], eixos[1]):
    eixo_hist.hist(serie, bins=35, color=AZUL, edgecolor=SUPERFICIE, linewidth=0.5)
    eixo_hist.set(title=f"{nome}: assimetria {serie.skew():.2f}", ylabel="Horas")
    (teoricos, amostra), (inclinacao, intercepto, _) = stats.probplot(serie)
    eixo_qq.scatter(teoricos, amostra, s=4, color=AZUL, alpha=0.4)
    eixo_qq.plot(teoricos, inclinacao * teoricos + intercepto, color=TINTA, linewidth=1.2)
    eixo_qq.set(xlabel="Quantis teóricos da normal", ylabel="Quantis da amostra")
plt.show()

_, lambda_box_cox = stats.boxcox(df_op[ALVO])
print(f"Lambda de Box-Cox estimado: {lambda_box_cox:.2f} (0 = log; 0,5 = raiz quadrada)")

In [ ]:
# Objetivo: Medir como a dispersão cresce com a média em cada combinação de estação e hora.
grupos = df_op.groupby(["Seasons", "Hour"])[ALVO].agg(["mean", "std"])
inclinacao_log, intercepto_log = np.polyfit(np.log(grupos["mean"]), np.log(grupos["std"]), 1)

fig, ax = plt.subplots(figsize=(6.2, 4.4))
ax.scatter(grupos["mean"], grupos["std"], s=22, color=AZUL, alpha=0.7)
medias_ordenadas = np.sort(grupos["mean"].to_numpy())
ax.plot(medias_ordenadas, np.exp(intercepto_log) * medias_ordenadas ** inclinacao_log,
        color=TINTA, linewidth=1.2)
ax.set(xscale="log", yscale="log", xlabel="Média da demanda no grupo (escala log)",
       ylabel="Desvio padrão no grupo (escala log)",
       title="A dispersão cresce na mesma proporção que a média")
plt.show()
print(f"Inclinação no gráfico log-log: {inclinacao_log:.2f}")

**O que observamos:**

- A **raiz quadrada** deixa a distribuição mais simétrica (assimetria 0,34). O **logaritmo** corrige
  demais e cria uma cauda à esquerda (−0,79). O lambda de Box-Cox fica entre os dois (0,28).
- Nenhuma escala produz normalidade: os *QQ-plots* mostram desvios nas caudas em todas elas. Isso é
  esperado, pois a demanda marginal mistura horas de perfis muito diferentes.
- O segundo gráfico traz outro ângulo: agrupando por estação e hora, a inclinação de 0,99 indica
  desvio padrão quase proporcional à média. Esse padrão é típico de **heterocedasticidade**, e a
  transformação que estabiliza a variância nesse caso é o logaritmo.

**Implicação para a Etapa 3:** forma e variância apontam para transformações diferentes. Em vez de
decidir agora, vale testar a raiz quadrada, o log e o Box-Cox, escolhendo pelos resíduos e pelo
desempenho na escala original. O que importa para a regressão é a forma dos **resíduos**, não da
variável resposta sozinha.

### 5.4 Tabelas de frequência das variáveis categóricas

Além das contagens, mostramos quantos **dias distintos** há em cada categoria. Como horas do mesmo
dia são muito parecidas, esse número indica melhor a quantidade real de informação.

In [ ]:
# Objetivo: Montar a tabela de frequência de uma categórica, com a contagem de dias distintos.
def tabela_frequencia(df, coluna):
    """Calcula horas, percentual e dias distintos em cada categoria.

    Parâmetros:
        df (DataFrame): dados com a coluna Date já convertida.
        coluna (str): nome da variável categórica.

    Retorna:
        DataFrame ordenado da categoria mais frequente para a menos frequente.
    """
    tabela = df.groupby(coluna, observed=True).agg(
        horas=("Date", "size"), dias_distintos=("Date", "nunique")
    )
    tabela["pct_horas"] = (100 * tabela["horas"] / len(df)).round(1)
    return tabela.sort_values("horas", ascending=False)


for categorica in ["Seasons", "Holiday", "Functioning_Day", "Day_Type"]:
    print(f"\n{categorica}")
    display(tabela_frequencia(df, categorica))

**O que observamos:**

- `Seasons` é equilibrada, com cerca de 25% em cada categoria, então as comparações entre estações
  têm boa base de dados.
- `Holiday` é a mais rara em termos práticos: as 348 horas parecem bastante, mas vêm de apenas
  **18 dias**. Um único feriado com clima ruim pode pesar muito nos resultados.
- `Functioning_Day = No` é rara (3,3%) e se concentra em poucos dias de primavera e outono.
- Em `Day_Type`, os dias úteis são cerca de dois terços dos dados.

## Seção 6 – Análise Bivariada: O que Move a Demanda?

### 6.1 Demanda versus variáveis contínuas

Cada ponto é uma hora. A linha escura é a **mediana por faixas**: como há muitos pontos
sobrepostos, ela deixa a tendência bem mais fácil de enxergar.

In [ ]:
# Objetivo: Definir a mediana por faixas e plotar a demanda contra cada variável contínua.
def mediana_por_faixas(x, y, n_faixas=15):
    """Calcula a mediana de y em faixas de x com quantidades de pontos parecidas.

    Parâmetros:
        x (Series): variável explicativa.
        y (Series): variável resposta.
        n_faixas (int): número máximo de faixas.

    Retorna:
        Tupla (centros, medianas) com as coordenadas da curva.
    """
    faixas = pd.qcut(x, n_faixas, duplicates="drop")
    agrupado = y.groupby(faixas, observed=True).median()
    centros = x.groupby(faixas, observed=True).median()
    return centros.to_numpy(), agrupado.to_numpy()


fig, eixos = plt.subplots(2, 4, figsize=(13, 6), sharey=True, constrained_layout=True)
for ax, coluna in zip(eixos.flat, METEOROLOGICAS):
    ax.scatter(df_op[coluna], df_op[ALVO], s=3, color=AZUL, alpha=0.12, rasterized=True)
    ax.plot(*mediana_por_faixas(df_op[coluna], df_op[ALVO]), color=TINTA, marker="o", ms=3)
    pearson = df_op[coluna].corr(df_op[ALVO])
    ax.set(title=f"{coluna}  (r = {pearson:.2f})", xlabel=coluna)
for ax in eixos[:, 0]:
    ax.set_ylabel("Bicicletas por hora")
plt.show()

In [ ]:
# Objetivo: Calcular a demanda por faixas de temperatura e de umidade para ler as tendências.
def resumir_por_faixas(df, coluna, limites):
    """Resume a demanda dentro de faixas definidas de uma variável contínua.

    Parâmetros:
        df (DataFrame): dados de origem.
        coluna (str): variável usada para criar as faixas.
        limites (list): limites das faixas.

    Retorna:
        DataFrame com média, mediana e quantidade de horas em cada faixa.
    """
    faixas = pd.cut(df[coluna], limites)
    return df.groupby(faixas, observed=True)[ALVO].agg(["mean", "median", "count"]).round(0)


display(resumir_por_faixas(df_op, "Temperature", [-20, -10, 0, 5, 10, 15, 20, 25, 30, 35, 40]))
display(resumir_por_faixas(df_op, "Humidity", [0, 20, 40, 60, 80, 100]))

**O que observamos:**

- **`Temperature`** é a relação mais clara: a mediana sobe de 115 bicicletas (abaixo de −10 °C) para
  mais de 1.100 (entre 25 e 30 °C) e **recua acima de 30 °C**. Essa curvatura sugere um termo
  quadrático no modelo.
- **`Dew_Point_Temperature`** tem curva parecida com a da temperatura, pois as duas se movem juntas.
- **`Humidity`** tem relação em arco: a demanda é maior entre 40% e 60% e despenca acima de 80%
  (mediana de 212). As horas com chuva têm umidade média de 90%, então parte dessa queda
  provavelmente é efeito da própria chuva.
- **`Rainfall`** e **`Snowfall`**: a maior diferença está entre *zero* e *não zero*. A partir daí,
  a demanda já é baixa e varia menos com o volume.
- **`Solar_Radiation`**: a massa de pontos no zero mistura noites movimentadas e madrugadas vazias,
  o que enfraquece a relação.
- **`Wind_Speed`** e **`Visibility`** têm relação fraca. O vento positivo (r = 0,12) provavelmente
  não é causal: ventos mais fortes ocorrem de dia, quando há mais gente na rua.

### 6.2 O ritmo do dia: hora e tipo de dia

O r = 0,42 entre `Hour` e a demanda sugere uma relação moderada. Mas será que uma reta descreve bem
a variação ao longo do dia?

In [ ]:
# Objetivo: Comparar o perfil horário da mediana de demanda entre os tipos de dia.
perfil = df_op.pivot_table(index="Hour", columns="Day_Type", values=ALVO, aggfunc="median")

fig, ax = plt.subplots(figsize=(10, 4.4))
for tipo in ORDEM_TIPO_DIA:
    ax.plot(perfil.index, perfil[tipo], color=CORES_TIPO_DIA[tipo], label=tipo)
for hora, texto in [(8, "8h: ida ao trabalho"), (18, "18h: volta")]:
    ax.annotate(texto, (hora, perfil.loc[hora, "Dia útil"]), xytext=(8, 2),
                textcoords="offset points", fontsize=9, color=TINTA_SECUNDARIA)
ax.set(xlabel="Hora do dia", ylabel="Mediana de bicicletas por hora", xticks=range(0, 24, 2),
       title="Dias úteis têm dois picos; fins de semana e feriados não")
ax.legend(loc="upper left")
plt.show()

print("Medianas por hora (seleção):")
display(perfil.loc[[4, 8, 13, 18, 21], ORDEM_TIPO_DIA].round(0))

In [ ]:
# Objetivo: Mostrar a mediana da demanda em cada combinação de mês e hora.
mapa_mes_hora = (
    df_op.pivot_table(index="Month", columns="Hour", values=ALVO, aggfunc="median")
    .loc[ORDEM_MESES]
)

fig, ax = plt.subplots(figsize=(11, 4.6))
sns.heatmap(mapa_mes_hora, cmap=MAPA_SEQUENCIAL, ax=ax, linewidths=0.4, linecolor=SUPERFICIE,
            cbar_kws={"label": "Mediana de bicicletas por hora"})
ax.set(xlabel="Hora do dia", ylabel="Mês (dez/2017 a nov/2018)",
       title="Picos das 8h e 18h em quase todo o ano; tarde mais fraca em julho e agosto")
ax.tick_params(axis="y", rotation=0)
plt.show()

**O que observamos:**

- A relação com a hora **não é uma reta**: há dois picos em dias úteis (cerca de 1.600 às 8h e
  1.900 às 18h), típicos de deslocamento casa–trabalho.
- Nos fins de semana, os picos desaparecem e a demanda sobe devagar até o fim da tarde. A madrugada
  de fim de semana também é **maior** que a de dias úteis.
- O mapa acrescenta um efeito da estação: em julho e agosto, as tardes ficam mais fracas que em
  junho e setembro, enquanto a noite segue forte. Isso combina com a queda acima de 30 °C vista em
  6.1, mas aqui é só uma associação.

**Implicação:** `Hour` deve entrar no modelo como categórica (ou por funções periódicas), e não como
número, e vale testar a **interação entre hora e tipo de dia**.

### 6.3 Visão completa do ano

Este mapa mostra cada hora de cada dia do dataset. As células em branco são as horas que não vieram neste
arquivo (provavelmente as do teste), e as marcas no topo indicam os dias em que o sistema esteve fechado em
algum momento.

In [ ]:
# Objetivo: Desenhar um mapa com todas as horas do ano, marcando os dias sem funcionamento.
datas_completas = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
mapa_ano = (
    df.pivot_table(index="Hour", columns="Date", values=ALVO, aggfunc="first")
    .reindex(columns=datas_completas)
)
dias_fechados = df.loc[df["Functioning_Day"] == "No", "Date"].unique()

mapa_cores = MAPA_SEQUENCIAL.copy()
mapa_cores.set_bad(SUPERFICIE)

fig, ax = plt.subplots(figsize=(12, 4.2))
malha = ax.pcolormesh(datas_completas, mapa_ano.index, np.ma.masked_invalid(mapa_ano.to_numpy()),
                      cmap=mapa_cores, shading="nearest", rasterized=True)
ax.scatter(dias_fechados, [24.3] * len(dias_fechados), marker="v", s=20, color=TINTA,
           clip_on=False, label="Dia com sistema fechado em alguma hora")
ax.set(ylim=(-0.5, 23.5), yticks=range(0, 24, 4), ylabel="Hora do dia")
ax.set_title("O ano inteiro: o inverno é fraco e as faixas das 8h e 18h aparecem claramente",
             pad=18)
ax.grid(False)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, -0.08))
fig.colorbar(malha, ax=ax, label="Bicicletas por hora", pad=0.015)
plt.show()

**O que observamos:** a imagem resume boa parte da análise: inverno fraco, primavera crescente,
faixas nos horários de pico e dias de fechamento concentrados no outono.

### 6.4 Comparação entre categorias

In [ ]:
# Objetivo: Comparar a distribuição da demanda entre estações e tipos de dia com violin plots.
def plotar_violinos(df, categoria, ordem, ax, titulo):
    """Desenha um violin plot da demanda por categoria, com os quartis marcados.

    Parâmetros:
        df (DataFrame): dados de origem.
        categoria (str): coluna categórica do eixo horizontal.
        ordem (list): ordem de exibição das categorias.
        ax (Axes): eixo onde desenhar.
        titulo (str): título do gráfico.

    Retorna:
        O próprio eixo recebido, já preenchido.
    """
    sns.violinplot(data=df, x=categoria, y=ALVO, order=ordem, ax=ax, color="#86b6ef", cut=0,
                   inner="quart", density_norm="width", linewidth=0.8)
    ax.set(title=titulo, xlabel="", ylabel="Bicicletas por hora")
    return ax


fig, eixos = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True, constrained_layout=True)
plotar_violinos(df_op, "Seasons", ORDEM_ESTACOES, eixos[0], "O inverno é outro patamar de demanda")
plotar_violinos(df_op, "Day_Type", ORDEM_TIPO_DIA, eixos[1],
                "Dias úteis têm a cauda mais longa, por causa dos picos")
plt.show()

medidas = ["mean", "median", "std", "count"]
display(df_op.groupby("Seasons")[ALVO].agg(medidas).loc[ORDEM_ESTACOES].round(0))
display(df_op.groupby("Day_Type")[ALVO].agg(medidas).loc[ORDEM_TIPO_DIA].round(0))

**O que observamos:**

- O **inverno** se destaca claramente: mediana de cerca de 200, contra 600 a 900 nas outras
  estações, além de dispersão bem menor.
- A diferença entre tipos de dia parece modesta quando comparamos apenas as distribuições gerais.
  A seção 6.2 mostrou por quê: fins de semana e dias úteis têm volumes parecidos, mas **distribuídos
  de forma diferente ao longo do dia**.

#### Feriados: um caso de variável de confusão

Feriados mostram mediana bem menor. Só que quase metade das horas de feriado ocorre no inverno,
quando a demanda já é baixa. Por isso, comparamos os grupos dentro de cada estação.

In [ ]:
# Objetivo: Comparar feriados e não feriados no total e dentro de cada estação.
mediana_geral = df_op.groupby("Holiday")[ALVO].median()
horas_por_estacao = pd.crosstab(df_op["Holiday"], df_op["Seasons"], normalize="index")
print(f"Mediana geral: feriado {mediana_geral['Holiday']:.0f} | "
      f"não feriado {mediana_geral['No Holiday']:.0f}")
print(f"Horas de feriado no inverno: {horas_por_estacao.loc['Holiday', 'Winter']:.0%}")
print(f"Horas de não feriado no inverno: {horas_por_estacao.loc['No Holiday', 'Winter']:.0%}")

feriado_estacao = df_op.pivot_table(index="Seasons", columns="Holiday", values=ALVO,
                                    aggfunc=["median", "count"]).loc[ORDEM_ESTACOES]
dias_feriado = df_op[df_op["Holiday"] == "Holiday"].groupby("Seasons")["Date"].nunique()
tabela_feriado = feriado_estacao.round(0).astype(int)
tabela_feriado[("dias_de_feriado", "")] = dias_feriado
display(tabela_feriado)

fig, ax = plt.subplots(figsize=(7.4, 3.8))
posicoes = np.arange(len(ORDEM_ESTACOES))
largura = 0.36
ax.bar(posicoes - largura / 2 - 0.01, feriado_estacao[("median", "No Holiday")], largura,
       color=CINZA, label="Não feriado")
ax.bar(posicoes + largura / 2 + 0.01, feriado_estacao[("median", "Holiday")], largura,
       color=VERDE, label="Feriado")
ax.set(xticks=posicoes, xticklabels=ORDEM_ESTACOES, ylabel="Mediana de bicicletas por hora",
       title="Dentro de cada estação, a diferença do feriado diminui ou desaparece")
ax.grid(axis="x", visible=False)
ax.legend()
plt.show()

**O que observamos:** olhando apenas o total, o feriado parece cortar a demanda mais da metade (de
560 para 250). Dentro das estações, a história muda: **no verão e no outono, as medianas são
praticamente iguais**; na primavera e no inverno, a queda é de cerca de um terço, contra 55% no
total. Boa parte do efeito bruto vem da composição: 47% das horas de feriado de operação estão no inverno, contra 24% nas demais
horas.

Como os feriados são só 18 dias, esses valores por estação ainda são instáveis (no verão, são
apenas 2 dias). Essa questão é uma boa candidata a teste na Etapa 2.

### 6.5 Chuva e neve

A correlação de Pearson entre chuva e demanda é só −0,14. Comparando as mesmas horas do dia, com e
sem chuva, o efeito fica bem mais claro.

In [ ]:
# Objetivo: Medir a queda de demanda com chuva ou neve, inclusive comparando a mesma hora do dia.
def comparar_ocorrencia(df, indicador, ax, rotulo, cor):
    """Plota o perfil horário da demanda mediana com e sem a ocorrência de um evento.

    Parâmetros:
        df (DataFrame): dados de origem.
        indicador (str): coluna binária (0 ou 1) que marca o evento.
        ax (Axes): eixo onde desenhar.
        rotulo (str): nome do evento, usado na legenda e no título.
        cor (str): cor da linha das horas com o evento.

    Retorna:
        DataFrame com mediana, média e número de horas nos dois grupos.
    """
    perfil_evento = df.pivot_table(index="Hour", columns=indicador, values=ALVO, aggfunc="median")
    ax.plot(perfil_evento.index, perfil_evento[0], color=CINZA, label=f"Sem {rotulo}")
    ax.plot(perfil_evento.index, perfil_evento[1], color=cor, label=f"Com {rotulo}")
    ax.set(title=f"Perfil horário com e sem {rotulo}", xlabel="Hora do dia",
           xticks=range(0, 24, 4))
    ax.legend(loc="upper left")
    return df.groupby(indicador)[ALVO].agg(["median", "mean", "count"]).round(0)


fig, eixos = plt.subplots(1, 2, figsize=(11.5, 4), sharey=True, constrained_layout=True)
resumo_chuva = comparar_ocorrencia(df_op, "Is_Rainy", eixos[0], "chuva", AZUL)
resumo_neve = comparar_ocorrencia(df_op, "Has_Snow", eixos[1], "neve", AZUL)
eixos[0].set_ylabel("Mediana de bicicletas por hora")
plt.show()

display(resumo_chuva)
display(resumo_neve)
display(resumir_por_faixas(df_op, "Rainfall", [-0.1, 0, 1, 5, 30]))

**O que observamos:**

- **Chuva tem um efeito forte em todas as horas:** a mediana cai de 591 para 70, e até o pico das
  18h praticamente some (cada hora tem poucos registros com chuva, então a linha azul é instável).
  A correlação é baixa porque apenas 6% das horas têm chuva e a relação não é linear: a maior parte
  da queda ocorre logo na faixa de até 1 mm, com mediana de 129.
- **A neve também reduz a demanda** (de 589 para 149), mas esse efeito se mistura com o do frio,
  já que a neve está quase toda no inverno.
- Assim, `Is_Rainy` tende a ser mais útil na regressão que o volume `Rainfall`, e essa é uma das
  comparações a fazer na Etapa 3.

### 6.6 Matriz de correlação

Calculamos as correlações de Pearson (associação linear) e Spearman (associação monotônica, baseada
em postos). A diferença entre elas ajuda a detectar relações não lineares e efeito de assimetria.

In [ ]:
# Objetivo: Plotar as matrizes de correlação de Pearson e Spearman lado a lado.
def plotar_correlacao(df, colunas, metodo, ax, com_barra=False):
    """Desenha a matriz de correlação anotada, com a metade superior oculta.

    Parâmetros:
        df (DataFrame): dados de origem.
        colunas (list): variáveis numéricas.
        metodo (str): "pearson" ou "spearman".
        ax (Axes): eixo onde desenhar.
        com_barra (bool): se deve desenhar a barra de cores.

    Retorna:
        DataFrame com a matriz de correlação.
    """
    matriz = df[colunas].corr(method=metodo)
    mascara = np.triu(np.ones_like(matriz, dtype=bool), k=1)
    sns.heatmap(matriz, mask=mascara, cmap=MAPA_DIVERGENTE, vmin=-1, vmax=1, annot=True, fmt=".2f",
                annot_kws={"size": 8}, ax=ax, cbar=com_barra, linewidths=0.5,
                linecolor=SUPERFICIE, square=True)
    ax.set_title(metodo.capitalize())
    ax.grid(False)
    return matriz


fig, eixos = plt.subplots(1, 2, figsize=(13.5, 6), constrained_layout=True)
pearson = plotar_correlacao(df_op, NUMERICAS, "pearson", eixos[0])
spearman = plotar_correlacao(df_op, NUMERICAS, "spearman", eixos[1], com_barra=True)
eixos[1].tick_params(labelleft=False)
plt.show()

com_alvo = pd.DataFrame({"pearson": pearson[ALVO], "spearman": spearman[ALVO]}).drop(ALVO)
com_alvo["diferenca_abs"] = (com_alvo["spearman"].abs() - com_alvo["pearson"].abs()).round(2)
display(com_alvo.round(2).sort_values("spearman", key=np.abs, ascending=False))

**Interpretação:**

- **Com a demanda:** `Temperature` lidera nas duas medidas (0,56 e 0,61). Em Pearson, vêm depois
  `Hour` (0,42) e `Dew_Point_Temperature` (0,40).
- **Spearman bem acima de Pearson em módulo:** `Rainfall` (−0,14 → −0,30), `Solar_Radiation` (0,28 →
  0,41) e `Snowfall` (−0,15 → −0,25). Isso indica relações fortes, porém não lineares, condizentes
  com o que vimos nos gráficos.
- **Entre as explicativas:** `Temperature` e `Dew_Point_Temperature` têm r = 0,92, e `Humidity`
  completa o trio ligado pela fórmula de Magnus. `Humidity` e `Visibility` têm correlação −0,55, e
  `Humidity` e `Solar_Radiation`, −0,46.

Correlação não implica causalidade: o vínculo entre vento e demanda, por exemplo, provavelmente é
efeito do horário.

### 6.7 Multicolinearidade

A correlação compara apenas duas variáveis por vez. O fator de inflação de variância (VIF) mede o
quanto cada variável pode ser explicada por **todas as outras juntas**. Valores acima de 10 são um
sinal comum de alerta para a regressão.

In [ ]:
# Objetivo: Definir o cálculo de VIF e comparar o conjunto completo com um conjunto reduzido.
def calcular_vif(df, colunas):
    """Calcula o fator de inflação de variância de cada coluna.

    Parâmetros:
        df (DataFrame): dados de origem.
        colunas (list): variáveis explicativas numéricas.

    Retorna:
        Series com o VIF de cada variável, em ordem decrescente.
    """
    matriz = df[colunas].astype(float).assign(constante=1.0).to_numpy()
    valores = [variance_inflation_factor(matriz, i) for i in range(len(colunas))]
    return pd.Series(valores, index=colunas, name="VIF").sort_values(ascending=False)


sem_orvalho = [c for c in METEOROLOGICAS if c != "Dew_Point_Temperature"]
tabela_vif = pd.DataFrame({
    "VIF (todas)": calcular_vif(df_op, METEOROLOGICAS),
    "VIF (sem ponto de orvalho)": calcular_vif(df_op, sem_orvalho),
}).sort_values("VIF (todas)", ascending=False)
display(tabela_vif.round(2))

**O que observamos:** com as três variáveis ligadas pela fórmula de Magnus, os VIFs de
`Dew_Point_Temperature` (195), `Temperature` (143) e `Humidity` (34) ficam muito acima de 10.
Retirando apenas o ponto de orvalho, todos os VIFs caem para menos de 2,5. Essa é uma justificativa concreta para a Etapa 3: o
ponto de orvalho adiciona pouca informação nova, porque é quase uma função das outras duas.

### 6.8 A associação com a temperatura muda conforme a estação?

Calculamos a correlação de Pearson entre a demanda e `Temperature` (e `Humidity`) separadamente
dentro de cada estação.

In [ ]:
# Objetivo: Medir como a associação da temperatura com a demanda muda dentro de cada estação.
por_estacao = pd.DataFrame({
    coluna: df_op.groupby("Seasons")[[coluna, ALVO]]
    .apply(lambda grupo: grupo[coluna].corr(grupo[ALVO]))
    for coluna in ["Temperature", "Humidity"]
}).loc[ORDEM_ESTACOES]
display(por_estacao.round(2))

**O que observamos:** a força da temperatura **varia entre as estações**: r = 0,57 na primavera e
0,17 no verão, possivelmente porque o calor deixa de estimular a demanda. Vale testar essa interação.

### 6.9 Dependência temporal

Muitos testes estatísticos supõem observações independentes. Calculamos a autocorrelação da demanda
em alguns intervalos para verificar se isso é razoável aqui.

In [ ]:
# Objetivo: Calcular a autocorrelação da demanda em intervalos de 1 hora, 1 dia e 1 semana.
instantes = df["Date"] + pd.to_timedelta(df["Hour"], unit="h")
serie_horaria = df.assign(instante=instantes).set_index("instante")[ALVO].sort_index().asfreq("h")

autocorrelacoes = pd.Series(
    {rotulo: serie_horaria.autocorr(lag) for rotulo, lag in
     {"1 hora": 1, "2 horas": 2, "24 horas": 24, "1 semana (168 h)": 168}.items()},
    name="autocorrelação",
)
display(autocorrelacoes.round(2).to_frame())

**O que observamos:** a autocorrelação é muito alta (0,90 com a hora anterior, e cerca de 0,65 a
0,68 no mesmo horário do dia ou da semana seguinte). As linhas **não são independentes**, e isso
tem duas consequências:

- **Etapa 2:** testes que tratam as horas como independentes tendem a produzir p-valores menores
  do que deveriam. Diferenças pequenas exigem cautela.
- **Etapa 3:** os resíduos devem ser avaliados também ao longo do tempo.

## Seção 7 – Fechamento: Variáveis Promissoras e Próximos Passos

### 7.1 Variáveis para o modelo

| Prioridade | Variável | Evidência | Como usar na Etapa 3 |
|---|---|---|---|
| **Alta** | `Hour` | r = 0,42, mas relação não linear; dois picos em dias úteis | Categórica ou funções periódicas |
| **Alta** | `Day_Type` (com `Hour`) | Fraca sozinha; muda completamente o perfil horário | Interação com `Hour` |
| **Alta** | `Temperature` | Maior correlação (0,56 / 0,61) e queda acima de 30 °C | Testar termo quadrático e interação com estação |
| **Alta** | `Is_Rainy` / `Rainfall` | Mediana cai de 591 para 70 | Comparar indicador binário com volume |
| **Alta** | `Functioning_Day` | Fechado implica demanda zero | Regra direta: previsão 0 quando fechado |
| **Média** | `Seasons` / `Month` | Inverno com mediana ~200 contra 600–900; `Month` detalha melhor | Comparar as duas, pois se sobrepõem à temperatura |
| **Média** | `Humidity` | Queda forte acima de 80% | Testar faixa ou termo não linear |
| **Média** | `Solar_Radiation` | Spearman 0,41, Pearson 0,28; zero à noite | Testar junto de `Hour`, com possível redundância |
| **Média** | `Has_Snow` | Mediana cai de 589 para 149 | Verificar se acrescenta algo além de temperatura e estação |
| **Baixa** | `Holiday` | Efeito menor dentro de cada estação; só 18 dias | Já representada em `Day_Type` |
| **Baixa** | `Wind_Speed`, `Visibility` | Associações fracas | Incluir e deixar a ANOVA avaliar |
| **Evitar** | `Dew_Point_Temperature` | VIF alto; quase determinada por temperatura e umidade | Retirar, salvo se melhorar a validação |

### 7.2 Hipóteses sugeridas para a Etapa 2

1. **Estações:** a demanda difere entre as estações? Como a variância muda muito entre os grupos, o
   teste de Kruskal-Wallis parece mais seguro que a ANOVA clássica, o que deve ser verificado.
2. **Feriados:** o efeito continua existindo depois de controlar pela estação? Esta hipótese
   aproveita diretamente o achado da seção 6.4.
3. **Chuva:** a demanda é menor nas horas com chuva, mesmo comparando a mesma hora do dia?

### 7.3 Cuidados para as próximas etapas

- Aplicar `preparar_dados` também no arquivo de teste, e nunca usar `df_op` para filtrar o teste: as
  linhas com sistema fechado precisam receber previsão (zero).
- Trazer as previsões de volta para a escala original, caso a resposta seja transformada.
- Lembrar que os dados cobrem **um único ano**. Os efeitos de estação e mês não podem ser separados
  de acontecimentos específicos de 2018, embora isso importe pouco para o teste, que vem do mesmo
  período.
- A associação encontrada aqui é descritiva: não demonstra causa, e a confirmação estatística fica
  para as próximas etapas.